# T = 1 Summary and Verification Runtime

This notebook combines the six evaluation ZIPs and three post-edit ZIPs. It generates the paper-style Figure 8, constructs the T = 1 entries of Tables 1 and 2, and reports runtime at the selected operating points.

The SPOT evaluation ZIPs contain only the final **SPOT-plugin** implementation based on the Li et al. optimal-weight fraction estimator, together with **SPOT-oracle**.

The runtime output separates:

- **pivot recomputation time**, measured from the saved final token sequences;
- **method time**, read from the evaluation ZIPs;
- **verification time**, defined as pivot recomputation time plus method time.

File loading, text editing, translation, decoding, re-tokenization, ground-truth construction, parameter sweeps, plotting, and output writing are excluded from the timing measurements.

The generated figure is saved as `figure8_T1.pdf` at 300 dpi.


In [ ]:
%pip -q install pandas matplotlib tqdm

In [ ]:
import json
import shutil
import tempfile
import time
import zipfile
from contextlib import contextmanager
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm


torch.set_grad_enabled(False)


In [ ]:
EVALUATION_ZIPS = {
    'random_SPOT': 'evaluation_random_SPOT_T1.zip',
    'random_AOL': 'evaluation_random_AOL_T1.zip',
    'adversarial_SPOT': 'evaluation_adversarial_SPOT_T1.zip',
    'adversarial_AOL': 'evaluation_adversarial_AOL_T1.zip',
    'roundtrip_SPOT': 'evaluation_roundtrip_SPOT_T1.zip',
    'roundtrip_AOL': 'evaluation_roundtrip_AOL_T1.zip',
}

POST_EDIT_ZIPS = {
    'random': 'post_edit_random_T1.zip',
    'adversarial': 'post_edit_adversarial_T1.zip',
    'roundtrip': 'post_edit_roundtrip_translation_T1.zip',
}

# None times every document. Set a positive integer for a faster timing estimate.
PIVOT_TIMING_MAX_DOCUMENTS = None

ROOT = Path('/content') if Path('/content').exists() else Path('/mnt/data')
OUTPUT_FOLDER = ROOT / 'T1_summary_outputs'
if OUTPUT_FOLDER.exists():
    shutil.rmtree(OUTPUT_FOLDER)
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)


def find_zip(filename: str) -> Path | None:
    for path in [Path('/content') / filename, Path('/mnt/data') / filename, Path.cwd() / filename]:
        if path.exists() and zipfile.is_zipfile(path):
            return path
    return None


def locate_required_zips(filenames) -> dict[str, Path]:
    filenames = list(dict.fromkeys(filenames))
    paths = {filename: find_zip(filename) for filename in filenames}
    missing = [filename for filename, path in paths.items() if path is None]

    if missing:
        try:
            from google.colab import files
            print('Upload the following ZIP files in one selection:')
            for filename in missing:
                print(' -', filename)
            files.upload()
        except Exception:
            pass
        paths = {filename: find_zip(filename) for filename in filenames}
        missing = [filename for filename, path in paths.items() if path is None]

    if missing:
        raise FileNotFoundError('Missing required ZIP files: ' + ', '.join(missing))
    return {filename: path for filename, path in paths.items() if path is not None}


def read_csv_from_zip(zip_path: Path, basename: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, 'r') as archive:
        matches = [name for name in archive.namelist() if Path(name).name == basename]
        if len(matches) != 1:
            raise FileNotFoundError(
                f'Expected one {basename} in {zip_path}; found {len(matches)}.'
            )
        return pd.read_csv(archive.open(matches[0]))


@contextmanager
def open_dataset_zip(zip_path: Path):
    with tempfile.TemporaryDirectory(prefix='spot_timing_', dir=str(ROOT)) as temporary_directory:
        temporary_path = Path(temporary_directory)
        with zipfile.ZipFile(zip_path, 'r') as archive:
            archive.extractall(temporary_path)

        dataset_paths = list(temporary_path.rglob('dataset.npz'))
        metadata_paths = list(temporary_path.rglob('meta.json'))
        if len(dataset_paths) != 1:
            raise FileNotFoundError(
                f'Expected one dataset.npz in {zip_path}; found {len(dataset_paths)}.'
            )
        if len(metadata_paths) != 1:
            raise FileNotFoundError(
                f'Expected one meta.json in {zip_path}; found {len(metadata_paths)}.'
            )

        data = np.load(dataset_paths[0], allow_pickle=False)
        metadata = json.loads(metadata_paths[0].read_text(encoding='utf-8'))
        try:
            yield data, metadata
        finally:
            data.close()


required_zip_paths = locate_required_zips([
    *EVALUATION_ZIPS.values(),
    *POST_EDIT_ZIPS.values(),
])
evaluation_paths = {
    label: required_zip_paths[filename] for label, filename in EVALUATION_ZIPS.items()
}
post_edit_paths = {
    label: required_zip_paths[filename] for label, filename in POST_EDIT_ZIPS.items()
}

selected_frames = []
runtime_frames = []
fraction_frames = []
for label, path in evaluation_paths.items():
    selected = read_csv_from_zip(path, 'selected_parameters.csv')
    selected['source'] = label
    selected_frames.append(selected)

    runtime = read_csv_from_zip(path, 'runtime_by_level.csv')
    runtime['source'] = label
    runtime_frames.append(runtime)

    if label.endswith('SPOT'):
        fractions = read_csv_from_zip(path, 'fraction_per_sample.csv')
        fractions['source'] = label
        fraction_frames.append(fractions)

selected_all = pd.concat(selected_frames, ignore_index=True)
runtime_all = pd.concat(runtime_frames, ignore_index=True)
fraction_all = pd.concat(fraction_frames, ignore_index=True)

print('Loaded six evaluation ZIPs and three post-edit ZIPs.')


## Figure 8

In [ ]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Times', 'DejaVu Serif'],
    'text.usetex': False,
})

MODE_ORDER = ['random_substitution', 'random_insertion', 'random_deletion']
MODE_XLABEL = {
    'random_substitution': 'Substitution fraction',
    'random_insertion': 'Insertion fraction',
    'random_deletion': 'Deletion fraction',
}
STYLE = {
    'SPOT-plugin': {
        'color': 'red', 'linestyle': '-', 'marker': 's',
    },
    'SPOT-oracle': {
        'color': 'blue', 'linestyle': '-', 'marker': '^',
    },
    'AOL': {
        'color': 'black', 'linestyle': '--', 'marker': 'o',
    },
}


def make_figure8(filename: str = 'figure8_T1.pdf') -> Path:
    plot_data = selected_all[
        (selected_all['case'].isin(MODE_ORDER))
        & (selected_all['method'].isin(['SPOT-plugin', 'SPOT-oracle', 'AOL']))
        & (selected_all['edit_level'] >= 0.10 - 1e-12)
    ].copy()

    fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.8), dpi=180)
    handles = {}
    for column, case in enumerate(MODE_ORDER):
        for row, (selection_metric, value_column, ylabel) in enumerate([
            ('IoU', 'selected_IoU', 'IoU'),
            ('TPR', 'selected_TPR', 'True positive rate'),
        ]):
            ax = axes[row, column]
            subset = plot_data[
                (plot_data['case'] == case)
                & (plot_data['selection_metric'] == selection_metric)
            ]
            for method in ['SPOT-plugin', 'SPOT-oracle', 'AOL']:
                method_data = subset[
                    subset['method'] == method
                ].sort_values('edit_level')
                if method_data.empty:
                    continue
                line, = ax.plot(
                    method_data['edit_level'],
                    method_data[value_column],
                    linewidth=2.3,
                    markersize=5.5,
                    label=method,
                    **STYLE[method],
                )
                handles[method] = line

            ax.set_xlim(0.10, 0.50)
            if case == 'random_deletion':
                ax.set_xticks([0.2, 0.3, 0.4, 0.5])
            else:
                ax.set_xticks([0.2, 0.4])
            if row == 0:
                ax.set_ylim(0.0, 0.50)
                ax.set_yticks([0.0, 0.2, 0.4])
            else:
                ax.set_ylim(0.0, 0.80)
                ax.set_yticks([0.0, 0.2, 0.4, 0.6, 0.8])
            ax.set_xlabel(MODE_XLABEL[case], fontsize=16)
            if column == 0:
                ax.set_ylabel(ylabel, fontsize=16)
            ax.grid(True, alpha=0.25)
            ax.tick_params(axis='both', labelsize=13)

    legend_order = ['SPOT-plugin', 'SPOT-oracle', 'AOL']
    fig.legend(
        [handles[name] for name in legend_order if name in handles],
        [name for name in legend_order if name in handles],
        loc='upper center',
        ncol=3,
        frameon=False,
        fontsize=16,
        bbox_to_anchor=(0.5, 1.01),
    )
    fig.subplots_adjust(
        left=0.08, right=0.99, bottom=0.09, top=0.89,
        wspace=0.28, hspace=0.30,
    )
    output_path = OUTPUT_FOLDER / filename
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print('Saved:', output_path)
    return output_path


make_figure8()


## Table 1: T = 1

In [ ]:
TABLE_LEVELS = {
    'random_substitution': [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40],
    'random_insertion': [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40],
    'random_deletion': [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40],
    'adversarial_edits': [10, 15, 20, 30, 40],
    'roundtrip_translation': None,
}

table1_rows = []
for case, levels in TABLE_LEVELS.items():
    for method in ['AOL', 'SPOT-plugin', 'SPOT-oracle']:
        subset = selected_all[
            (selected_all['case'] == case)
            & (selected_all['method'] == method)
        ]
        if levels is not None:
            level_values = subset['edit_level'].to_numpy(dtype=float)
            subset = subset[
                np.isclose(
                    level_values[:, None],
                    np.asarray(levels, dtype=float),
                ).any(axis=1)
            ]
        iou_values = subset[
            subset['selection_metric'] == 'IoU'
        ]['selected_IoU']
        tpr_values = subset[
            subset['selection_metric'] == 'TPR'
        ]['selected_TPR']
        if iou_values.empty or tpr_values.empty:
            raise RuntimeError(f'Missing Table 1 results for {case}, {method}.')
        table1_rows.append({
            'edit_type': case,
            'method': method,
            'IoU_T1': float(iou_values.mean()),
            'TPR_T1': float(tpr_values.mean()),
            'n_levels': int(max(len(iou_values), len(tpr_values))),
        })

table1 = pd.DataFrame(table1_rows)
table1.to_csv(OUTPUT_FOLDER / 'table1_T1.csv', index=False)
display(table1)


## Table 2: T = 1 fraction-estimation accuracy


In [ ]:
def relative_rmse_by_level(frame: pd.DataFrame) -> pd.DataFrame:
    records = []
    for group_values, group in frame.groupby(
        ['case', 'level_index', 'edit_level'], dropna=False, sort=True
    ):
        case, level_index, edit_level = group_values
        true_mean = float(group['epsilon_true'].mean())
        value = float(
            np.sqrt(np.mean((group['epsilon_plugin'] - group['epsilon_true']) ** 2))
            / (true_mean + 1e-12)
        )
        records.append({
            'case': case,
            'level_index': level_index,
            'edit_level': edit_level,
            'relative_RMSE': value,
        })
    return pd.DataFrame(records)


by_level = relative_rmse_by_level(fraction_all)
table2_rows = []
for case, levels in TABLE_LEVELS.items():
    subset = by_level[by_level['case'] == case]
    if levels is not None:
        level_values = subset['edit_level'].to_numpy(dtype=float)
        subset = subset[
            np.isclose(
                level_values[:, None],
                np.asarray(levels, dtype=float),
            ).any(axis=1)
        ]
    if subset.empty:
        raise RuntimeError(f'Missing Table 2 results for {case}.')
    table2_rows.append({
        'edit_type': case,
        'fraction_estimator': 'SPOT-plugin',
        'relative_RMSE_T1': float(subset['relative_RMSE'].mean()),
        'n_levels': int(len(subset)),
    })

table2 = pd.DataFrame(table2_rows)
table2.to_csv(OUTPUT_FOLDER / 'table2_T1.csv', index=False)
display(table2)


## Pivot recomputation timing

The timing code uses the same Gumbel-pivot reconstruction as the post-edit notebooks. It reads the saved post-edit token sequences and prompt tokens, verifies the reconstruction against a saved pivot array, and then times pivot recomputation without file I/O.

The timed verifier windows are:

- random substitution and insertion: 400 positions;
- random deletion: 200 positions;
- adversarial edits: 400 positions;
- roundtrip translation: the last 200 positions.

The random-edit and adversarial timing summaries use the same edit levels averaged in Tables 1 and 2.


In [ ]:
_rng_table = torch.Generator(device='cpu')
_rng_table.manual_seed(2971215073)
_TABLE_SIZE = 1_000_003
_FIXED_TABLE = torch.randperm(_TABLE_SIZE, device='cpu', generator=_rng_table)


def _hashint(integer_tensor: torch.LongTensor) -> torch.LongTensor:
    return _FIXED_TABLE[integer_tensor.cpu() % _TABLE_SIZE] + 1


def _noncomm_prf(input_ids: torch.LongTensor, salt_key: int) -> int:
    key_value = torch.as_tensor(int(salt_key), dtype=torch.long)
    for entry in input_ids:
        key_value *= _hashint(key_value * entry)
        key_value %= 2**32 - 1
    return int(key_value.item())


def _seed_rng(generator: torch.Generator, tokens_1xL: torch.LongTensor,
              seeding_scheme: str, hash_key: int, context_width: int) -> None:
    if seeding_scheme != 'noncomm_prf':
        raise ValueError("This notebook supports seeding_scheme='noncomm_prf'.")
    if tokens_1xL.shape[-1] < context_width:
        raise ValueError('The prefix is shorter than the watermark context width.')
    seed = _noncomm_prf(tokens_1xL[0, -context_width:], salt_key=hash_key)
    generator.manual_seed(seed)


def recompute_gumbel_pivots_window(
    text_ids: np.ndarray,
    prompt_ids: np.ndarray,
    vocab_size: int,
    key: int,
    context_width: int,
    seeding_scheme: str,
    start_index: int = 0,
    end_index: int | None = None,
) -> np.ndarray:
    text = torch.as_tensor(np.asarray(text_ids, dtype=np.int64), dtype=torch.long)
    sequence_length = int(text.numel())
    if end_index is None:
        end_index = sequence_length
    start_index = int(start_index)
    end_index = int(end_index)
    if not 0 <= start_index <= end_index <= sequence_length:
        raise ValueError(
            f'Invalid pivot window [{start_index}, {end_index}) for length {sequence_length}.'
        )

    generator = torch.Generator(device='cpu')
    prompt_tail = torch.as_tensor(
        np.asarray(prompt_ids, dtype=np.int64)[-context_width:], dtype=torch.long
    )
    full_sequence = torch.cat([prompt_tail, text], dim=0)

    pivots = np.empty(end_index - start_index, dtype=np.float32)
    for output_index, position in enumerate(range(start_index, end_index)):
        prefix = full_sequence[:context_width + position].unsqueeze(0)
        _seed_rng(generator, prefix, seeding_scheme, key, context_width)
        xi = torch.rand((vocab_size,), generator=generator)
        token_id = int(full_sequence[context_width + position].item())
        pivots[output_index] = (
            float(xi[token_id].item()) if 0 <= token_id < vocab_size else np.nan
        )
    return pivots


def pivot_parameters(metadata: dict) -> tuple[int, int, int, str]:
    vocab_size = int(metadata.get('vocab_size_full', 0))
    if vocab_size <= 0:
        raise ValueError('meta.json must contain a positive vocab_size_full.')
    key = int(metadata.get('key', 15485863))
    context_width = int(metadata.get('c_window', 5))
    seeding_scheme = str(metadata.get('seeding_scheme', 'noncomm_prf'))
    return vocab_size, key, context_width, seeding_scheme


def find_level_index(values: np.ndarray, target: float) -> int:
    matches = np.flatnonzero(np.isclose(values, target, rtol=0.0, atol=1e-12))
    if len(matches) != 1:
        raise ValueError(f'Expected one match for level {target}; found {len(matches)}.')
    return int(matches[0])


def timing_document_indices(num_documents: int) -> np.ndarray:
    if PIVOT_TIMING_MAX_DOCUMENTS is None:
        return np.arange(num_documents, dtype=np.int32)
    maximum = int(PIVOT_TIMING_MAX_DOCUMENTS)
    if maximum <= 0:
        raise ValueError('PIVOT_TIMING_MAX_DOCUMENTS must be None or a positive integer.')
    return np.arange(min(maximum, num_documents), dtype=np.int32)


def time_pivot_group(
    *,
    case: str,
    level_index: int,
    edit_level: float,
    token_matrix: np.ndarray,
    saved_pivot_matrix: np.ndarray,
    prompt_matrix: np.ndarray,
    metadata: dict,
    start_index: int,
    end_index: int,
) -> dict:
    token_matrix = np.asarray(token_matrix)
    saved_pivot_matrix = np.asarray(saved_pivot_matrix)
    prompt_matrix = np.asarray(prompt_matrix)
    if token_matrix.ndim != 2 or saved_pivot_matrix.shape != token_matrix.shape:
        raise ValueError(
            f'Invalid token/pivot shapes for {case}: '
            f'{token_matrix.shape} and {saved_pivot_matrix.shape}.'
        )
    if prompt_matrix.shape[0] != token_matrix.shape[0]:
        raise ValueError(f'Prompt count does not match document count for {case}.')

    vocab_size, key, context_width, seeding_scheme = pivot_parameters(metadata)
    document_indices = timing_document_indices(token_matrix.shape[0])
    if document_indices.size == 0:
        raise ValueError(f'No documents available for {case}.')

    # Untimed reconstruction check. This also warms up the reconstruction path.
    first_document = int(document_indices[0])
    reconstructed = recompute_gumbel_pivots_window(
        token_matrix[first_document],
        prompt_matrix[first_document],
        vocab_size,
        key,
        context_width,
        seeding_scheme,
        start_index,
        end_index,
    )
    expected = np.asarray(
        saved_pivot_matrix[first_document, start_index:end_index], dtype=np.float32
    )
    if not np.allclose(reconstructed, expected, rtol=0.0, atol=1e-7, equal_nan=True):
        maximum_difference = float(np.nanmax(np.abs(reconstructed - expected)))
        raise RuntimeError(
            f'Pivot reconstruction check failed for {case}, level {edit_level}. '
            f'Maximum absolute difference: {maximum_difference}.'
        )

    document_times = []
    description = f'Pivot timing: {case}, level={edit_level}'
    for document_index in tqdm(document_indices, desc=description, leave=False):
        started = time.perf_counter()
        recompute_gumbel_pivots_window(
            token_matrix[int(document_index)],
            prompt_matrix[int(document_index)],
            vocab_size,
            key,
            context_width,
            seeding_scheme,
            start_index,
            end_index,
        )
        document_times.append(time.perf_counter() - started)

    document_times = np.asarray(document_times, dtype=float)
    return {
        'case': case,
        'level_index': int(level_index),
        'edit_level': float(edit_level) if np.isfinite(edit_level) else np.nan,
        'sequence_length': int(token_matrix.shape[1]),
        'evaluation_start_index': int(start_index),
        'evaluation_length': int(end_index - start_index),
        'pivot_runtime_seconds_total_timed': float(document_times.sum()),
        'pivot_runtime_seconds_per_document': float(document_times.mean()),
        'pivot_runtime_seconds_median_per_document': float(np.median(document_times)),
        'pivot_runtime_seconds_std_per_document': float(document_times.std(ddof=0)),
        'n_documents_available': int(token_matrix.shape[0]),
        'n_documents_timed': int(document_indices.size),
        'full_dataset_timing': bool(document_indices.size == token_matrix.shape[0]),
    }


pivot_records = []

# Random substitution, insertion, and deletion.
with open_dataset_zip(post_edit_paths['random']) as (data, metadata):
    required = {
        'prompts_tokens',
        'tokens_post_sub_repo', 'Ys_post_sub_repo',
        'tokens_post_ist_repo', 'Ys_post_ist_repo',
        'tokens_post_dlt_repo', 'Ys_post_dlt_repo',
    }
    missing = required.difference(data.files)
    if missing:
        raise KeyError(f'Missing random post-edit arrays: {sorted(missing)}')
    edit_rates = np.asarray(
        metadata.get('edit_rates', np.arange(0.0, 0.50 + 1e-12, 0.05)), dtype=float
    )
    prompts = np.asarray(data['prompts_tokens'])
    random_specs = {
        'random_substitution': ('tokens_post_sub_repo', 'Ys_post_sub_repo'),
        'random_insertion': ('tokens_post_ist_repo', 'Ys_post_ist_repo'),
        'random_deletion': ('tokens_post_dlt_repo', 'Ys_post_dlt_repo'),
    }
    for case, (token_key, pivot_key) in random_specs.items():
        tokens = np.asarray(data[token_key])
        saved_pivots = np.asarray(data[pivot_key])
        for edit_level in TABLE_LEVELS[case]:
            level_index = find_level_index(edit_rates, edit_level)
            pivot_records.append(time_pivot_group(
                case=case,
                level_index=level_index,
                edit_level=edit_level,
                token_matrix=tokens[level_index],
                saved_pivot_matrix=saved_pivots[level_index],
                prompt_matrix=prompts,
                metadata=metadata,
                start_index=0,
                end_index=tokens.shape[-1],
            ))

# Adversarial edits.
with open_dataset_zip(post_edit_paths['adversarial']) as (data, metadata):
    required = {'prompts_tokens'}
    for budget in TABLE_LEVELS['adversarial_edits']:
        required.update({
            f'tokens_post_adv_repo_k{budget}',
            f'Ys_post_adv_repo_k{budget}',
        })
    missing = required.difference(data.files)
    if missing:
        raise KeyError(f'Missing adversarial post-edit arrays: {sorted(missing)}')
    budgets = np.asarray(metadata.get('top_k_list', [5, 10, 15, 20, 30, 40]), dtype=float)
    prompts = np.asarray(data['prompts_tokens'])
    for budget in TABLE_LEVELS['adversarial_edits']:
        level_index = find_level_index(budgets, float(budget))
        tokens = np.asarray(data[f'tokens_post_adv_repo_k{budget}'])
        saved_pivots = np.asarray(data[f'Ys_post_adv_repo_k{budget}'])
        pivot_records.append(time_pivot_group(
            case='adversarial_edits',
            level_index=level_index,
            edit_level=float(budget),
            token_matrix=tokens,
            saved_pivot_matrix=saved_pivots,
            prompt_matrix=prompts,
            metadata=metadata,
            start_index=0,
            end_index=tokens.shape[-1],
        ))

# Roundtrip translation: match the last-200 evaluation window.
with open_dataset_zip(post_edit_paths['roundtrip']) as (data, metadata):
    required = {'prompts_tokens', 'tokens_post_trans_repo', 'Ys_post_trans_repo'}
    missing = required.difference(data.files)
    if missing:
        raise KeyError(f'Missing roundtrip post-edit arrays: {sorted(missing)}')
    prompts = np.asarray(data['prompts_tokens'])
    tokens = np.asarray(data['tokens_post_trans_repo'])
    saved_pivots = np.asarray(data['Ys_post_trans_repo'])
    evaluation_window = 200
    if tokens.shape[-1] < evaluation_window:
        raise ValueError('Roundtrip sequences are shorter than the 200-position evaluation window.')
    start_index = tokens.shape[-1] - evaluation_window
    pivot_records.append(time_pivot_group(
        case='roundtrip_translation',
        level_index=0,
        edit_level=np.nan,
        token_matrix=tokens,
        saved_pivot_matrix=saved_pivots,
        prompt_matrix=prompts,
        metadata=metadata,
        start_index=start_index,
        end_index=tokens.shape[-1],
    ))

pivot_runtime_by_level = pd.DataFrame(pivot_records)
pivot_runtime_by_level.to_csv(
    OUTPUT_FOLDER / 'pivot_runtime_by_level_T1.csv', index=False
)
display(pivot_runtime_by_level)

pivot_timing_metadata = {
    'pivot_timing_max_documents': PIVOT_TIMING_MAX_DOCUMENTS,
    'timing_scope': 'Evaluation-window Gumbel-pivot recomputation from saved post-edit token sequences.',
    'excluded': [
        'ZIP loading', 'editing', 'translation', 'decoding', 're-tokenization',
        'ground-truth construction', 'parameter sweeps', 'plotting', 'output writing',
    ],
    'roundtrip_evaluation_window': 200,
}
(OUTPUT_FOLDER / 'pivot_timing_config.json').write_text(
    json.dumps(pivot_timing_metadata, indent=2), encoding='utf-8'
)


## Runtime at the selected operating points

In [ ]:
# Retain the same edit levels used in the paper's runtime summaries.
method_runtime_parts = []
for case, levels in TABLE_LEVELS.items():
    subset = runtime_all[runtime_all['case'] == case].copy()
    if levels is not None:
        level_values = subset['edit_level'].to_numpy(dtype=float)
        subset = subset[
            np.isclose(level_values[:, None], np.asarray(levels, dtype=float)).any(axis=1)
        ]
    method_runtime_parts.append(subset)

method_runtime_by_level = pd.concat(method_runtime_parts, ignore_index=True).rename(columns={
    'runtime_seconds_total': 'method_runtime_seconds_total',
    'runtime_seconds_per_document': 'method_runtime_seconds_per_document',
    'n_documents': 'method_n_documents',
})

pivot_columns = [
    'case', 'level_index', 'edit_level', 'sequence_length',
    'evaluation_start_index', 'evaluation_length',
    'pivot_runtime_seconds_total_timed',
    'pivot_runtime_seconds_per_document',
    'pivot_runtime_seconds_median_per_document',
    'pivot_runtime_seconds_std_per_document',
    'n_documents_available', 'n_documents_timed', 'full_dataset_timing',
]

runtime_by_level = method_runtime_by_level.merge(
    pivot_runtime_by_level[pivot_columns],
    on=['case', 'level_index'],
    how='left',
    validate='many_to_one',
    suffixes=('', '_pivot'),
)
if runtime_by_level['pivot_runtime_seconds_per_document'].isna().any():
    missing_cases = runtime_by_level.loc[
        runtime_by_level['pivot_runtime_seconds_per_document'].isna(),
        ['case', 'level_index'],
    ].drop_duplicates()
    raise RuntimeError(
        'Missing pivot timing for some evaluation rows:\n' + missing_cases.to_string(index=False)
    )

runtime_by_level['verification_runtime_seconds_per_document'] = (
    runtime_by_level['pivot_runtime_seconds_per_document']
    + runtime_by_level['method_runtime_seconds_per_document']
)
runtime_by_level['estimated_pivot_runtime_seconds_total_for_method_documents'] = (
    runtime_by_level['pivot_runtime_seconds_per_document']
    * runtime_by_level['method_n_documents']
)
runtime_by_level['estimated_verification_runtime_seconds_total'] = (
    runtime_by_level['estimated_pivot_runtime_seconds_total_for_method_documents']
    + runtime_by_level['method_runtime_seconds_total']
)
runtime_by_level.to_csv(OUTPUT_FOLDER / 'runtime_by_level_T1.csv', index=False)

runtime_rows = []
for case, levels in TABLE_LEVELS.items():
    case_frame = runtime_by_level[runtime_by_level['case'] == case]
    for (method, selection_metric), group in case_frame.groupby(
        ['method', 'selection_metric'], sort=True
    ):
        subset = group
        if levels is not None:
            level_values = subset['edit_level'].to_numpy(dtype=float)
            subset = subset[
                np.isclose(level_values[:, None], np.asarray(levels, dtype=float)).any(axis=1)
            ]
        if subset.empty:
            continue
        runtime_rows.append({
            'edit_type': case,
            'method': method,
            'selection_metric': selection_metric,
            'mean_pivot_runtime_seconds_per_document': float(
                subset['pivot_runtime_seconds_per_document'].mean()
            ),
            'mean_method_runtime_seconds_per_document': float(
                subset['method_runtime_seconds_per_document'].mean()
            ),
            'mean_verification_runtime_seconds_per_document': float(
                subset['verification_runtime_seconds_per_document'].mean()
            ),
            'mean_method_runtime_seconds_total_per_level': float(
                subset['method_runtime_seconds_total'].mean()
            ),
            'mean_estimated_verification_runtime_seconds_total_per_level': float(
                subset['estimated_verification_runtime_seconds_total'].mean()
            ),
            'n_levels': int(len(subset)),
            'minimum_pivot_timing_documents': int(subset['n_documents_timed'].min()),
            'all_pivot_timings_use_full_dataset': bool(subset['full_dataset_timing'].all()),
        })

runtime_summary = pd.DataFrame(runtime_rows)
runtime_summary.to_csv(OUTPUT_FOLDER / 'runtime_summary_T1.csv', index=False)
display(runtime_summary)


## Save the summary ZIP

In [ ]:
summary_zip = ROOT / 'T1_summary_outputs.zip'
if summary_zip.exists():
    summary_zip.unlink()
shutil.make_archive(str(summary_zip.with_suffix('')), 'zip', root_dir=OUTPUT_FOLDER)
print('Saved:', summary_zip)
try:
    from google.colab import files
    files.download(str(summary_zip))
except Exception:
    pass
